In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("/Users/vconklin24/Desktop/pen_america_banned_books.csv")

In [5]:
df

,Title,Author,Secondary Author(s),Illustrator(s),Translator(s),Series,State,District,Date of Challenge/Removal,Ban Status
0,Nineteen Minutes,"Picoult, Jodi",NaN,NaN,NaN,NaN,Arizona,Higley Unified School District,August 2024,Banned by Restriction
1,The Perks of Being a Wallflower,"Chbosky, Stephen",NaN,NaN,NaN,NaN,Arizona,Higley Unified School District,August 2024,Banned by Restriction
2,#Pride: Championing LGBTQ Rights,"Felix, Rebecca",NaN,NaN,NaN,#movements,Colorado,Elizabeth School District,August 2024,Banned
3,Beloved,"Morrison, Toni",NaN,NaN,NaN,NaN,Colorado,Elizabeth School District,August 2024,Banned
4,Burned (EH),"Hopkins, Ellen",NaN,NaN,NaN,Burned,Colorado,Elizabeth School District,August 2024,Banned
...,...,...,...,...,...,...,...,...,...,...
6714,Fallout,"Hopkins, Ellen",NaN,NaN,NaN,Crank,Wyoming,Laramie County School District No. 1,August 2024,Banned by Restriction
6715,Georgia Peaches and Other Forbidden Fruit,"Brown, Jaye Robin",NaN,NaN,NaN,NaN,Wyoming,Laramie County School District No. 1,September 2024,Banned by Restriction
6716,How Beautiful the Ordinary: Twelve Stories of ...,"Cart, Michael",NaN,NaN,NaN,NaN,Wyoming,Laramie County School District No. 1,September 2024,Banned by Restriction
6717,Perfect (EH),"Hopkins, Ellen",NaN,NaN,NaN,Impulse,Wyoming,Laramie County School District No. 1,September 2024,Banned by Restriction


In [7]:
"""
Goodreads Book ID Finder
Searches Goodreads to find book IDs for scraping
Author: Violet (Goodreads Specialist)
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm
import logging

# Set up logging
logging.basicConfig(
    filename='book_id_search.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def search_goodreads_book(book_title, author=None, retries=3):
    """
    Search Goodreads for a book and return its ID and URL
    
    Parameters:
    - book_title: String, the book title to search
    - author: String (optional), author name for more specific search
    - retries: Number of times to retry if request fails
    
    Returns:
    - Dictionary with book_id, url, and found status
    """
    
    # Construct search query
    search_query = book_title
    if author:
        search_query += f" {author}"
    
    search_url = f"https://www.goodreads.com/search?q={search_query.replace(' ', '+')}"
    
    for attempt in range(retries):
        try:
            # Set headers to mimic browser
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
            
            response = requests.get(search_url, headers=headers, timeout=10)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Find first search result
            first_result = soup.find('a', class_='bookTitle')
            
            if first_result:
                book_url = "https://www.goodreads.com" + first_result['href']
                
                # Extract book ID from URL
                # URL format: /book/show/[ID]-title or /book/show/[ID].title
                book_id = book_url.split('/show/')[1].split('-')[0].split('.')[0]
                
                logging.info(f"Found {book_title}: ID={book_id}")
                
                return {
                    'book_title': book_title,
                    'author': author,
                    'goodreads_id': book_id,
                    'goodreads_url': book_url,
                    'found': True,
                    'search_query': search_query
                }
            else:
                logging.warning(f"No results found for: {book_title}")
                return {
                    'book_title': book_title,
                    'author': author,
                    'goodreads_id': None,
                    'goodreads_url': None,
                    'found': False,
                    'search_query': search_query
                }
                
        except Exception as e:
            if attempt < retries - 1:
                logging.warning(f"Attempt {attempt + 1} failed for {book_title}: {e}. Retrying...")
                time.sleep(5)
            else:
                logging.error(f"All attempts failed for {book_title}: {e}")
                return {
                    'book_title': book_title,
                    'author': author,
                    'goodreads_id': None,
                    'goodreads_url': None,
                    'found': False,
                    'search_query': search_query,
                    'error': str(e)
                }

def find_all_book_ids(input_csv='Grid.csv.csv', output_csv='data/book_ids/goodreads_ids.csv'):
    """
    Find Goodreads IDs for all books in the dataset
    
    Parameters:
    - input_csv: Path to PEN America CSV
    - output_csv: Path to save book IDs
    """
    # Read input data
    books_df = pd.read_csv(input_csv)
    print(f"Searching for {len(books_df)} books on Goodreads...")
    
    # Initialize results list
    all_results = []
    
    # Search for each book
    for index, row in tqdm(books_df.iterrows(), total=len(books_df), desc="Searching"):
        book_title = row.get('Book Title', row.get('Title', ''))
        author = row.get('Author', None)
        
        result = search_goodreads_book(book_title, author)
        all_results.append(result)
        
        # Be respectful - wait between requests
        time.sleep(3)
        
        # Save progress every 10 books
        if (index + 1) % 10 == 0:
            temp_df = pd.DataFrame(all_results)
            temp_df.to_csv(output_csv.replace('.csv', '_temp.csv'), index=False)
    
    # Save final results
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(output_csv, index=False)
    
    # Print summary
    found_count = results_df['found'].sum()
    print(f"\n=== SEARCH COMPLETE ===")
    print(f"Books found: {found_count}/{len(results_df)}")
    print(f"Books not found: {len(results_df) - found_count}")
    print(f"Results saved to: {output_csv}")
    
    # Save list of books not found
    not_found = results_df[~results_df['found']]
    if len(not_found) > 0:
        not_found.to_csv('documentation/books_not_found.csv', index=False)
        print(f"Books not found saved to: documentation/books_not_found.csv")
    
    return results_df

# Main execution
if __name__ == "__main__":
    # Test with a few books first
    print("Testing with sample books...")
    test_books = [
        ("Gender Queer", "Maia Kobabe"),
        ("All Boys Aren't Blue", "George M. Johnson"),
        ("The Bluest Eye", "Toni Morrison")
    ]
    
    for title, author in test_books:
        result = search_goodreads_book(title, author)
        print(f"{title}: {result}")
        time.sleep(3)
    
    # Uncomment to run full search in Week 1
    # print("\nRunning full search...")
    # find_all_book_ids()

Testing with sample books...
Gender Queer: {'book_title': 'Gender Queer', 'author': 'Maia Kobabe', 'goodreads_id': '150896123', 'goodreads_url': 'https://www.goodreads.com/book/show/150896123-study-guide?from_search=true&from_srp=true&qid=rTyXXDlyNZ&rank=1', 'found': True, 'search_query': 'Gender Queer Maia Kobabe'}
All Boys Aren't Blue: {'book_title': "All Boys Aren't Blue", 'author': 'George M. Johnson', 'goodreads_id': '121281417', 'goodreads_url': 'https://www.goodreads.com/book/show/121281417-study-guide?from_search=true&from_srp=true&qid=F4gyHtmdjD&rank=1', 'found': True, 'search_query': "All Boys Aren't Blue George M. Johnson"}
The Bluest Eye: {'book_title': 'The Bluest Eye', 'author': 'Toni Morrison', 'goodreads_id': '45294414', 'goodreads_url': 'https://www.goodreads.com/book/show/45294414-study-guide?from_search=true&from_srp=true&qid=dD3GGrb5oz&rank=1', 'found': True, 'search_query': 'The Bluest Eye Toni Morrison'}
